# Nhánh NER thứ hai — PhoBERT trên VietMed-NER

**Notebook release: `2026-09-09-v9-full-validation-seeds`**

Notebook duy nhất chạy tuần tự: 18 cấu hình validation (learning rate × epoch × weight decay), 3 seed cho cấu hình tốt nhất, gold benchmark và pipeline ASR trên 500 audio. Test chỉ được đọc ở các cell đánh giá cuối. Chọn GPU T4 trước khi chạy.

In [1]:
!pip -q install -U "transformers>=4.41,<5" "datasets>=2.19,<4" accelerate safetensors seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 87.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 83.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
import json
from pathlib import Path
from typing import Any

import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments,
)

BASE_MODEL = 'vinai/phobert-base-v2'
DATASET_ID = 'leduckhai/VietMed-NER'
OUTPUT_DIR = Path('/content/phobert-vietmed-ner')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED = 42


def to_jsonable(value):
    """Convert NumPy/Torch scalar and nested report values to JSON types."""
    if isinstance(value, dict):
        return {str(key): to_jsonable(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_jsonable(item) for item in value]
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().tolist()
    return value

print('Device:', DEVICE)
print('Dataset:', DATASET_ID)
print('Base model:', BASE_MODEL)

Device: cuda
Dataset: leduckhai/VietMed-NER
Base model: vinai/phobert-base-v2


## 1. Tải dữ liệu và chuẩn bị nhãn

Dataset được tải trực tiếp từ Hugging Face. Notebook tự hỗ trợ cả cột nhãn `labels` của dataset hiện tại và `tags` của phiên bản cũ.

In [3]:
dataset = load_dataset(DATASET_ID)
label_column = 'labels' if 'labels' in dataset['train'].column_names else 'tags'
feature = dataset['train'].features[label_column].feature
MAX_LENGTH = 256

# Label vocabulary is derived from train only; test is not touched during setup or tuning.
if hasattr(feature, 'names'):
    label_names = list(feature.names)
else:
    label_names = sorted({
        str(label)
        for example in dataset['train']
        for label in example[label_column]
    })

label2id = {label: index for index, label in enumerate(label_names)}
id2label = {index: label for label, index in label2id.items()}

def label_to_id(label: Any) -> int:
    if isinstance(label, int) and hasattr(feature, 'names'):
        label = feature.int2str(label)
    label = str(label)
    if label not in label2id:
        raise ValueError(f'Unknown label {label!r}; expected {label_names}')
    return label2id[label]

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
print('Tokenizer:', tokenizer.__class__.__name__)
print('Tokenizer fast:', getattr(tokenizer, 'is_fast', False))

def encode_words(words, current_tokenizer, row_labels=None):
    raw_ids = []
    raw_word_ids = []
    raw_labels = [] if row_labels is not None else None
    for word_index, word in enumerate(words):
        subtoken_ids = current_tokenizer.encode(word, add_special_tokens=False)
        if not subtoken_ids:
            continue
        raw_ids.extend(subtoken_ids)
        raw_word_ids.extend([word_index] * len(subtoken_ids))
        if raw_labels is not None:
            raw_labels.extend([label_to_id(row_labels[word_index])] + [-100] * (len(subtoken_ids) - 1))

    input_ids = current_tokenizer.build_inputs_with_special_tokens(raw_ids)
    raw_start = None
    for start in range(len(input_ids) - len(raw_ids) + 1):
        if input_ids[start:start + len(raw_ids)] == raw_ids:
            raw_start = start
            break
    if raw_start is None:
        raise ValueError('Could not locate raw tokens in special-token sequence')

    input_ids = input_ids[:MAX_LENGTH]
    raw_end = min(raw_start + len(raw_ids), len(input_ids))
    aligned_word_ids = [None] * len(input_ids)
    aligned_labels = [-100] * len(input_ids) if raw_labels is not None else None
    for raw_index, position in enumerate(range(raw_start, raw_end)):
        aligned_word_ids[position] = raw_word_ids[raw_index]
        if aligned_labels is not None:
            aligned_labels[position] = raw_labels[raw_index]

    return {
        'input_ids': input_ids,
        'attention_mask': [1] * len(input_ids),
    }, aligned_word_ids, aligned_labels

def tokenize_and_align(examples):
    batch = {'input_ids': [], 'attention_mask': [], 'labels': []}
    for words, row_labels in zip(examples['words'], examples[label_column]):
        encoded, _, aligned_labels = encode_words(words, tokenizer, row_labels)
        for key in encoded:
            batch[key].append(encoded[key])
        batch['labels'].append(aligned_labels)
    return batch

tokenized = {
    split_name: dataset[split_name].map(
        tokenize_and_align,
        batched=True,
        remove_columns=dataset[split_name].column_names,
    )
    for split_name in ('train', 'validation')
}
smoke_example = tokenized['train'][0]
assert len(smoke_example['input_ids']) == len(smoke_example['attention_mask']) == len(smoke_example['labels'])
assert any(label != -100 for label in smoke_example['labels']), 'No aligned gold labels found'
print('Tokenization smoke test: passed')
print('Tuning splits:', {name: len(split) for name, split in tokenized.items()})
print('Test split is reserved for the final evaluation cells.')
print('Label column:', label_column)
print('Labels:', label_names)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/442M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/111M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/340M [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer: PhobertTokenizer
Tokenizer fast: False


Map:   0%|          | 0/4616 [00:00<?, ? examples/s]

Map:   0%|          | 0/1154 [00:00<?, ? examples/s]

Tokenization smoke test: passed
Tuning splits: {'train': 4616, 'validation': 1154}
Test split is reserved for the final evaluation cells.
Label column: labels
Labels: ['0', 'B-AGE', 'B-DATETIME', 'B-DIAGNOSTICS', 'B-DISEASESYMTOM', 'B-DRUGCHEMICAL', 'B-FOODDRINK', 'B-GENDER', 'B-LOCATION', 'B-MEDDEVICETECHNIQUE', 'B-OCCUPATION', 'B-ORGAN', 'B-ORGANIZATION', 'B-PERSONALCARE', 'B-PREVENTIVEMED', 'B-SURGERY', 'B-TRANSPORTATION', 'B-TREATMENT', 'B-UNITCALIBRATOR', 'I-AGE', 'I-DATETIME', 'I-DIAGNOSTICS', 'I-DISEASESYMTOM', 'I-DRUGCHEMICAL', 'I-FOODDRINK', 'I-GENDER', 'I-LOCATION', 'I-MEDDEVICETECHNIQUE', 'I-OCCUPATION', 'I-ORGAN', 'I-ORGANIZATION', 'I-PERSONALCARE', 'I-PREVENTIVEMED', 'I-SURGERY', 'I-TRANSPORTATION', 'I-TREATMENT', 'I-UNITCALIBRATOR']


## 2. Fine-tune PhoBERT

Checkpoint được lưu tại `/content/phobert-vietmed-ner`. Có thể giảm `NUM_TRAIN_EPOCHS` hoặc batch size nếu GPU không đủ bộ nhớ.

In [4]:
# 2. Sweep 18 cấu hình trên train/validation, sau đó 3 seed cho best config
import gc
import itertools
import time
from statistics import mean, pstdev

from seqeval.metrics import f1_score, precision_score, recall_score


def compute_metrics(eval_prediction):
    predictions, labels = eval_prediction
    predicted_ids = np.argmax(predictions, axis=2)
    true_predictions, true_labels = [], []
    for prediction, label_row in zip(predicted_ids, labels):
        pred_tags, gold_tags = [], []
        for predicted_id, gold_id in zip(prediction, label_row):
            if gold_id == -100:
                continue
            pred_tags.append(id2label[int(predicted_id)])
            gold_tags.append(id2label[int(gold_id)])
        true_predictions.append(pred_tags)
        true_labels.append(gold_tags)
    return {
        'precision': float(precision_score(true_labels, true_predictions, zero_division=0)),
        'recall': float(recall_score(true_labels, true_predictions, zero_division=0)),
        'f1': float(f1_score(true_labels, true_predictions, zero_division=0)),
    }


def build_trainer(model, output_dir, learning_rate, epochs, weight_decay, seed):
    training_kwargs = dict(
        output_dir=str(output_dir), learning_rate=learning_rate,
        per_device_train_batch_size=16, per_device_eval_batch_size=16,
        num_train_epochs=epochs, weight_decay=weight_decay,
        lr_scheduler_type='linear', warmup_ratio=0.1,
        save_strategy='epoch', save_total_limit=1,
        load_best_model_at_end=True, metric_for_best_model='f1',
        greater_is_better=True, logging_steps=50, seed=seed,
        report_to='none',
    )
    import inspect
    if 'eval_strategy' in inspect.signature(TrainingArguments.__init__).parameters:
        training_kwargs['eval_strategy'] = 'epoch'
    else:
        training_kwargs['evaluation_strategy'] = 'epoch'
    trainer_args = TrainingArguments(**training_kwargs)
    trainer_kwargs = dict(
        model=model, args=trainer_args,
        train_dataset=tokenized['train'], eval_dataset=tokenized['validation'],
        data_collator=DataCollatorForTokenClassification(tokenizer),
        compute_metrics=compute_metrics,
    )
    if 'processing_class' in inspect.signature(Trainer.__init__).parameters:
        trainer_kwargs['processing_class'] = tokenizer
    else:
        trainer_kwargs['tokenizer'] = tokenizer
    return Trainer(**trainer_kwargs)


def train_trial(trial_id, learning_rate, epochs, weight_decay, seed):
    trial_dir = OUTPUT_DIR / trial_id
    trial_dir.mkdir(parents=True, exist_ok=True)
    started = time.time()
    model = AutoModelForTokenClassification.from_pretrained(
        BASE_MODEL, num_labels=len(label_names), id2label=id2label, label2id=label2id
    )
    trainer = build_trainer(model, trial_dir, learning_rate, epochs, weight_decay, seed)
    trainer.train()
    metrics = to_jsonable(trainer.evaluate(tokenized['validation']))
    trainer.save_model(str(trial_dir))
    tokenizer.save_pretrained(str(trial_dir))
    result = {
        'trial_id': trial_id, 'learning_rate': learning_rate, 'epochs': epochs,
        'weight_decay': weight_decay, 'warmup_ratio': 0.1,
        'scheduler': 'linear', 'seed': seed, 'validation': metrics,
        'checkpoint': str(trial_dir), 'seconds': round(time.time() - started, 2),
    }
    del trainer, model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result

learning_rates = [5e-6, 1e-5, 3e-5]
epoch_options = [4, 6, 8]
weight_decays = [0.01, 0.05]
trial_specs = [
    (f'validation_lr{lr:.0e}_ep{epochs}_wd{wd}', lr, epochs, wd)
    for lr, epochs, wd in itertools.product(learning_rates, epoch_options, weight_decays)
]
sweep_path = OUTPUT_DIR / 'validation_trials.jsonl'
completed_trials = {}
if sweep_path.exists():
    for line in sweep_path.read_text(encoding='utf-8').splitlines():
        if line.strip():
            item = json.loads(line)
            completed_trials[item['trial_id']] = item
for trial_id, lr, epochs, wd in trial_specs:
    if trial_id in completed_trials:
        print('Resume: skip', trial_id)
        continue
    result = train_trial(trial_id, lr, epochs, wd, seed=SEED)
    completed_trials[trial_id] = result
    with sweep_path.open('a', encoding='utf-8') as handle:
        handle.write(json.dumps(to_jsonable(result), ensure_ascii=False) + '\n')
    print('Completed', trial_id, 'validation F1=', result['validation']['eval_f1'])

trial_results = sorted(completed_trials.values(), key=lambda item: item['validation']['eval_f1'], reverse=True)
assert len(trial_results) == len(trial_specs), f'Expected 18 trials, got {len(trial_results)}'
best_config = trial_results[0]
(OUTPUT_DIR / 'validation_sweep.json').write_text(
    json.dumps(to_jsonable({'trials': trial_results, 'best_config': best_config}), ensure_ascii=False, indent=2),
    encoding='utf-8',
)
print('Best validation config:', json.dumps(to_jsonable(best_config), ensure_ascii=False, indent=2))

seed_path = OUTPUT_DIR / 'best_config_seeds.jsonl'
seed_results = {}
if seed_path.exists():
    for line in seed_path.read_text(encoding='utf-8').splitlines():
        if line.strip():
            item = json.loads(line)
            seed_results[item['seed']] = item
for seed in (42, 123, 2024):
    if seed in seed_results:
        print('Resume: skip seed', seed)
        continue
    seed_id = f"seed_{seed}_lr{best_config['learning_rate']:.0e}_ep{best_config['epochs']}_wd{best_config['weight_decay']}"
    result = train_trial(
        seed_id, best_config['learning_rate'], best_config['epochs'],
        best_config['weight_decay'], seed,
    )
    seed_results[seed] = result
    with seed_path.open('a', encoding='utf-8') as handle:
        handle.write(json.dumps(to_jsonable(result), ensure_ascii=False) + '\n')
    print('Completed seed', seed, 'validation F1=', result['validation']['eval_f1'])

seed_results_list = sorted(seed_results.values(), key=lambda item: item['validation']['eval_f1'], reverse=True)
assert len(seed_results_list) == 3
seed_f1_values = [item['validation']['eval_f1'] for item in seed_results_list]
best_seed = seed_results_list[0]
seed_summary = {
    'best_config': best_config,
    'seeds': seed_results_list,
    'validation_f1_mean': mean(seed_f1_values),
    'validation_f1_std_population': pstdev(seed_f1_values),
    'selected_seed': best_seed,
}
(OUTPUT_DIR / 'best_config_seeds.json').write_text(
    json.dumps(to_jsonable(seed_summary), ensure_ascii=False, indent=2), encoding='utf-8'
)
print('Seed validation mean/std:', seed_summary['validation_f1_mean'], seed_summary['validation_f1_std_population'])
print('Selected seed checkpoint:', best_seed['checkpoint'])

pytorch_model.bin:   0%|          | 0.00/540M [00:00<?, ?B/s]

Some weights of RobertaForTokenClassification were not initialized from the model checkpoint at vinai/phobert-base-v2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


model.safetensors:   0%|          | 0.00/540M [00:00<?, ?B/s]

Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,1.374800,1.239644,0.000000,0.000000,0.000000
2,1.091300,1.028793,0.320321,0.176054,0.227222
3,0.969800,0.929796,0.392898,0.262835,0.314968
4,0.941600,0.898337,0.432820,0.317816,0.366508


/usr/local/lib/python3.13/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: 0 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.13/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: 0 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.13/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: 0 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.13/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: 0 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))


...

Some weights of RobertaForTokenClassification were not initialized from the model checkpoint at vinai/phobert-base-v2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.913200,0.704280,0.550501,0.431226,0.483618
2,0.393500,0.349727,0.729353,0.751149,0.740091
3,0.268000,0.257537,0.767616,0.784674,0.776052
4,0.187200,0.214008,0.803002,0.830077,0.816315
5,0.145400,0.204092,0.807260,0.826437,0.816736
6,0.128800,0.194196,0.814877,0.837356,0.825964
7,0.088900,0.191205,0.815234,0.842720,0.828749
8,0.080900,0.191839,0.819060,0.844636,0.831651


/usr/local/lib/python3.13/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: 0 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.13/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: 0 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.13/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: 0 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.13/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: 0 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.13/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: 0 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.13/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: 

/usr/local/lib/python3.13/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: 0 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))


Completed seed 2024 validation F1= 0.8316514194096011
Seed validation mean/std: 0.8343954211266398 0.0036171274992010554
Selected seed checkpoint: /content/phobert-vietmed-ner/seed_123_lr3e-05_ep8_wd0.05


## 3. So sánh PhoBERT với baseline trên cùng test split

In [5]:
# 3. Gold evaluation: dùng checkpoint seed tốt nhất theo validation, test chỉ chạy tại đây
best_model = AutoModelForTokenClassification.from_pretrained(best_seed['checkpoint']).to(DEVICE)
baseline_model = AutoModelForTokenClassification.from_pretrained(
    'leduckhai/VietMed-NER', subfolder='xlm-roberta-base-VietMed-NER'
).to(DEVICE)
baseline_tokenizer = AutoTokenizer.from_pretrained(
    'leduckhai/VietMed-NER', subfolder='xlm-roberta-base-VietMed-NER'
)

from seqeval.metrics import classification_report, f1_score, precision_score, recall_score

def evaluate_checkpoint(eval_model, eval_tokenizer, examples):
    eval_model.eval()
    model_id2label = {int(key): value for key, value in eval_model.config.id2label.items()}
    predictions, references = [], []
    for example in examples:
        encoded, aligned_word_ids, _ = encode_words(example['words'], eval_tokenizer)
        model_inputs = {
            key: torch.tensor([value], dtype=torch.long, device=DEVICE)
            for key, value in encoded.items()
        }
        with torch.inference_mode():
            logits = eval_model(**model_inputs).logits
        predicted_ids = logits.argmax(dim=-1)[0].detach().cpu().tolist()
        row_predictions, row_references = [], []
        seen_words = set()
        for token_id, word_id in zip(predicted_ids, aligned_word_ids):
            if word_id is None or word_id in seen_words:
                continue
            if word_id >= len(example[label_column]):
                break
            seen_words.add(word_id)
            row_predictions.append(model_id2label[int(token_id)])
            gold = example[label_column][word_id]
            if isinstance(gold, int) and hasattr(feature, 'names'):
                gold = feature.int2str(gold)
            row_references.append(str(gold))
        predictions.append(row_predictions)
        references.append(row_references)
    report = classification_report(references, predictions, output_dict=True, zero_division=0)
    return to_jsonable({
        'precision': float(precision_score(references, predictions, zero_division=0)),
        'recall': float(recall_score(references, predictions, zero_division=0)),
        'f1': float(f1_score(references, predictions, zero_division=0)),
        'classification_report': report,
        'num_examples': len(examples),
    })

gold_metrics = {
    'baseline_xlm_roberta': evaluate_checkpoint(baseline_model, baseline_tokenizer, dataset['test']),
    'phobert_best_validation_seed': evaluate_checkpoint(best_model, tokenizer, dataset['test']),
}
gold_payload = {
    'dataset': DATASET_ID, 'split': 'test',
    'best_config': best_config, 'seed_summary': seed_summary,
    'metrics': gold_metrics,
    'note': 'Both models use the same test split, manual first-subtoken alignment and seqeval evaluator. PhoBERT checkpoint was selected using validation only; test was run once here.',
}
Path('/content/ner_gold_comparison.json').write_text(
    json.dumps(to_jsonable(gold_payload), ensure_ascii=False, indent=2), encoding='utf-8'
)
for name, values in gold_metrics.items():
    print(f"{name}: P={values['precision']:.4f} R={values['recall']:.4f} F1={values['f1']:.4f} n={values['num_examples']}")

config.json: 0.00B [00:00, ?B/s]

xlm-roberta-base-VietMed-NER/pytorch_mod(…):   0%|          | 0.00/1.11G [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

xlm-roberta-base-VietMed-NER/sentencepie(…):   0%|          | 0.00/5.07M [00:00<?, ?B/s]

xlm-roberta-base-VietMed-NER/tokenizer.j(…):   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

baseline_xlm_roberta: P=0.5182 R=0.6278 F1=0.5678 n=3497
phobert_best_validation_seed: P=0.5620 R=0.6646 F1=0.6090 n=3497


In [7]:
!pip install jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 98.1 MB/s eta 0:00:00


In [8]:
# 4. ASR → hai NER trên cùng 500 audio/test
from collections import Counter
from datasets import Audio
from jiwer import wer
from transformers import (
    AutoFeatureExtractor,
    AutoModelForSpeechSeq2Seq,
    AutoProcessor,
    pipeline,
)

ASR_REPO = 'leduckhai/MultiMed-ST'
ASR_SUBFOLDER = 'asr/whisper-small-vietnamese/checkpoint-5000'
ASR_PROCESSOR_SUBFOLDER = 'asr/whisper-small-vietnamese'
ASR_SAMPLES = 500
ASR_OUTPUT = Path('/content/asr_ner_comparison.jsonl')

if not torch.cuda.is_available():
    raise RuntimeError('ASR evaluation requires a CUDA GPU. Select a T4 runtime.')

asr_model = AutoModelForSpeechSeq2Seq.from_pretrained(
    ASR_REPO,
    subfolder=ASR_SUBFOLDER,
    dtype=torch.float16,
    low_cpu_mem_usage=True,
    use_safetensors=True,
).to('cuda')

# Depending on the Hub bundle, AutoProcessor may return a full processor or a tokenizer directly.
asr_processor = AutoProcessor.from_pretrained(
    ASR_REPO, subfolder=ASR_PROCESSOR_SUBFOLDER
)
asr_tokenizer = getattr(asr_processor, 'tokenizer', asr_processor)
asr_feature_extractor = getattr(asr_processor, 'feature_extractor', None)
if asr_feature_extractor is None:
    asr_feature_extractor = AutoFeatureExtractor.from_pretrained(
        ASR_REPO, subfolder=ASR_PROCESSOR_SUBFOLDER
    )
print('ASR tokenizer:', type(asr_tokenizer).__name__)
print('ASR feature extractor:', type(asr_feature_extractor).__name__)

asr_pipe = pipeline(
    'automatic-speech-recognition',
    model=asr_model,
    tokenizer=asr_tokenizer,
    feature_extractor=asr_feature_extractor,
    dtype=torch.float16,
    device=0,
)

baseline_ner_pipe = pipeline(
    'ner', model=baseline_model, tokenizer=baseline_tokenizer,
    aggregation_strategy='simple', device=0,
)
phobert_ner_pipe = pipeline(
    'ner', model=best_model, tokenizer=tokenizer,
    aggregation_strategy='simple', device=0,
)

audio_dataset = load_dataset('leduckhai/VietMed', split='test')
audio_dataset = audio_dataset.cast_column(
    'audio', Audio(sampling_rate=16000, decode=True)
)

def extract_entities(ner_pipe, text):
    entities = []
    for item in ner_pipe(text):
        label = item.get('entity_group') or item.get('entity')
        if label in {'0', 'O', 'dum'}:
            continue
        entities.append({'text': item['word'], 'label': label})
    return entities

def entity_counter(entities):
    return Counter(
        (item['text'].strip().lower(), item['label']) for item in entities
    )

asr_records = []
num_samples = min(ASR_SAMPLES, len(audio_dataset))
for index, row in enumerate(audio_dataset.select(range(num_samples))):
    audio = row['audio']
    asr_result = asr_pipe(
        {'raw': audio['array'], 'sampling_rate': audio['sampling_rate']},
        generate_kwargs={'language': 'Vietnamese', 'task': 'transcribe'},
    )
    transcript = asr_result['text'].strip()
    reference = row.get('text', '') or ''

    baseline_reference = extract_entities(baseline_ner_pipe, reference)
    baseline_asr = extract_entities(baseline_ner_pipe, transcript)
    phobert_reference = extract_entities(phobert_ner_pipe, reference)
    phobert_asr = extract_entities(phobert_ner_pipe, transcript)

    baseline_ref = entity_counter(baseline_reference)
    baseline_asr_keys = entity_counter(baseline_asr)
    phobert_ref = entity_counter(phobert_reference)
    phobert_asr_keys = entity_counter(phobert_asr)
    asr_records.append({
        'index': index,
        'reference_transcript': reference,
        'asr_transcript': transcript,
        'wer': float(wer(reference, transcript)) if reference and transcript else None,
        'baseline_reference_entities': baseline_reference,
        'baseline_asr_entities': baseline_asr,
        'phobert_reference_entities': phobert_reference,
        'phobert_asr_entities': phobert_asr,
        'baseline_matched_entities': sum((baseline_ref & baseline_asr_keys).values()),
        'baseline_reference_entity_count': sum(baseline_ref.values()),
        'phobert_matched_entities': sum((phobert_ref & phobert_asr_keys).values()),
        'phobert_reference_entity_count': sum(phobert_ref.values()),
    })
    if (index + 1) % 25 == 0:
        print(f'ASR/NER completed: {index + 1}/{num_samples}')

ASR_OUTPUT.write_text(
    ''.join(json.dumps(to_jsonable(row), ensure_ascii=False) + '\n' for row in asr_records),
    encoding='utf-8',
)

def asr_summary(prefix):
    matched = sum(row[f'{prefix}_matched_entities'] for row in asr_records)
    reference_count = sum(row[f'{prefix}_reference_entity_count'] for row in asr_records)
    return {
        'matched_entities': matched,
        'reference_entity_count': reference_count,
        'entity_retention': matched / reference_count if reference_count else None,
    }

asr_summary_payload = {
    'num_samples': len(asr_records),
    'asr_model': f'{ASR_REPO}/{ASR_SUBFOLDER}',
    'baseline_xlm_roberta': asr_summary('baseline'),
    'phobert_best_validation': asr_summary('phobert'),
    'note': 'Both NER models process the identical ASR transcript. Retention compares each model prediction on reference vs ASR text and is not gold recall.',
}
Path('/content/asr_ner_comparison_summary.json').write_text(
    json.dumps(to_jsonable(asr_summary_payload), ensure_ascii=False, indent=2),
    encoding='utf-8',
)
print(json.dumps(to_jsonable(asr_summary_payload), ensure_ascii=False, indent=2))

config.json: 0.00B [00:00, ?B/s]

asr/whisper-small-vietnamese/checkpoint-(…):   0%|          | 0.00/967M [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Generating dev split:   0%|          | 0/2912 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3437 [00:00<?, ? examples/s]

Generating cv split:   0%|          | 0/85 [00:00<?, ? examples/s]

`return_token_timestamps` is deprecated for WhisperFeatureExtractor and will be removed in Transformers v5. Use `return_attention_mask` instead, as the number of frames can be inferred from it.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


ASR/NER completed: 25/500
ASR/NER completed: 50/500
ASR/NER completed: 75/500
ASR/NER completed: 100/500
ASR/NER completed: 125/500
ASR/NER completed: 150/500
ASR/NER completed: 175/500
ASR/NER completed: 200/500
ASR/NER completed: 225/500
ASR/NER completed: 250/500
ASR/NER completed: 275/500
ASR/NER completed: 300/500
ASR/NER completed: 325/500
ASR/NER completed: 350/500
ASR/NER completed: 375/500
ASR/NER completed: 400/500
ASR/NER completed: 425/500
ASR/NER completed: 450/500
ASR/NER completed: 475/500
ASR/NER completed: 500/500
{
  "num_samples": 500,
  "asr_model": "leduckhai/MultiMed-ST/asr/whisper-small-vietnamese/checkpoint-5000",
  "baseline_xlm_roberta": {
    "matched_entities": 785,
    "reference_entity_count": 1487,
    "entity_retention": 0.5279085406859448
  },
  "phobert_best_validation": {
    "matched_entities": 741,
    "reference_entity_count": 1391,
    "entity_retention": 0.5327102803738317
  },
  "note": "Both NER models process the identical ASR transcript. Rete

In [9]:
from google.colab import files
import json
from pathlib import Path

# Đóng gói checkpoint seed tốt nhất (nhẹ hơn zip toàn bộ 21 trial)
seed_manifest = Path('/content/phobert-vietmed-ner/best_config_seeds.json')
best_checkpoint = None
if seed_manifest.exists():
    data = json.loads(seed_manifest.read_text(encoding='utf-8'))
    best_checkpoint = data.get('selected_seed', {}).get('checkpoint')

if best_checkpoint:
    !zip -qr /content/phobert-best-seed.zip "{best_checkpoint}"
    best_zip = '/content/phobert-best-seed.zip'
else:
    # Fallback: đóng gói toàn bộ nếu chưa có seeds
    !zip -qr /content/phobert-vietmed-ner-full.zip /content/phobert-vietmed-ner
    best_zip = '/content/phobert-vietmed-ner-full.zip'

to_download = [
    '/content/ner_gold_comparison.json',
    '/content/asr_ner_comparison.jsonl',
    '/content/asr_ner_comparison_summary.json',
    '/content/phobert-vietmed-ner/validation_trials.jsonl',
    '/content/phobert-vietmed-ner/validation_sweep.json',
    '/content/phobert-vietmed-ner/best_config_seeds.jsonl',
    '/content/phobert-vietmed-ner/best_config_seeds.json',
    best_zip,
]

for path in to_download:
    if Path(path).exists():
        print('Tải:', path)
        files.download(path)
    else:
        print('Không tồn tại:', path)

Tải: /content/ner_gold_comparison.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Tải: /content/asr_ner_comparison.jsonl


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Tải: /content/phobert-best-seed.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# 5. Tải toàn bộ artifact benchmark về máy
!zip -qr /content/phobert-vietmed-ner-full.zip /content/phobert-vietmed-ner
from google.colab import files
for output_path in [
    '/content/ner_gold_comparison.json',
    '/content/asr_ner_comparison.jsonl',
    '/content/asr_ner_comparison_summary.json',
    '/content/phobert-vietmed-ner/validation_trials.jsonl',
    '/content/phobert-vietmed-ner/validation_sweep.json',
    '/content/phobert-vietmed-ner/best_config_seeds.jsonl',
    '/content/phobert-vietmed-ner/best_config_seeds.json',
    '/content/phobert-vietmed-ner-full.zip',
]:
    files.download(output_path)